# 02 — Preprocess to MedSAM2 NPZ format

Converts raw volumes (TIFF stacks from Chalmers, possibly TIFF or NIfTI stacks from ACDC) to the MedSAM2 on-disk format: a single `.npz` per volume with keys `imgs` (Z, H, W, uint8) and `gts` (Z, H, W, uint8 class ids).

For the demo we generate a labelled synthetic dataset from `cfrp_medsam2.synthetic` — same strategy as the Chalmers paper (train on synthetic, evaluate on real).

Output structure:
```
data/processed/
  toy/            synthetic train/val volumes with labels
  cfrp/           real chunks with dummy zero labels (unsupervised eval only)
  sic_sic/        real CMC volumes (user provides labels via `preprocess.ingest_tiff_stack`)
```

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import tifffile
from cfrp_medsam2.data import percentile_normalize, save_npz_volume
from cfrp_medsam2.synthetic import ToyVolumeConfig, make_toy_dataset

REPO = Path('..').resolve()
RAW = REPO / 'data' / 'raw'
PROC = REPO / 'data' / 'processed'
(PROC / 'toy').mkdir(parents=True, exist_ok=True)
(PROC / 'cfrp').mkdir(parents=True, exist_ok=True)

## 1. Synthetic labelled volumes for training
We generate volumes with ~80 tightly-packed fibres, mimicking the 3D-textile CFRP microstructure, and save a 3-train / 1-val split.

In [ ]:
cfg = ToyVolumeConfig(shape=(48, 256, 256), num_fibres=80, fibre_radius=(2.5, 4.0))
data = make_toy_dataset(n_train=3, n_val=1, cfg=cfg)
for i, (imgs, gts) in enumerate(data['train']):
    save_npz_volume(PROC / 'toy' / f'train_{i:02d}.npz', imgs, gts)
for i, (imgs, gts) in enumerate(data['val']):
    save_npz_volume(PROC / 'toy' / f'val_{i:02d}.npz', imgs, gts)
list((PROC / 'toy').glob('*.npz'))

## 2. Real CFRP chunk for zero-shot qualitative eval (no GT)

In [ ]:
real_tiff = RAW / 'cfrp' / 'real_orthogonal_noobed_sample_reconstruction.tiff'
if real_tiff.exists():
    real = tifffile.imread(real_tiff)
    real_u8 = percentile_normalize(real)
    z0 = (real_u8.shape[0] - 64) // 2
    sub = real_u8[z0:z0 + 64]
    zero_gt = np.zeros_like(sub, dtype=np.uint8)
    save_npz_volume(PROC / 'cfrp' / 'real_chunk.npz', sub, zero_gt)
    print('wrote', PROC / 'cfrp' / 'real_chunk.npz', sub.shape)
else:
    print('no real scan on disk — skipping')

## 3. SiC-SiC (optional, if the ACDC archive was fetched manually)
Once you've pulled the paired image / label TIFF stacks from ACDC into `data/raw/sic_sic/`, run:

```python
from cfrp_medsam2.preprocess import ingest_directory
ingest_directory(
    RAW / 'sic_sic' / 'images',
    RAW / 'sic_sic' / 'labels',
    PROC / 'sic_sic',
    label_lut={0: 0, 128: 1, 255: 2},  # adjust to your label palette
    resize=512,
)
```